# Day 4: Oscillation Parameter Explorer

This notebook turns the Day 2 spectrum into a parameter study. You will use `nuosclab` to ask: if θ₂₃ or Δm²₃₂ were different, what would change in the data?

By the end you should be able to:
- connect Δm²₃₂ to the dip position,
- connect θ₂₃ to the dip depth,
- explain why statistical fluctuations complicate a measurement,
- build a simple scan that finds which parameter values best match a toy dataset.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from nuosclab import ExplorerConfig, PMNSParams, compute_curves
from nuosclab.presets import PRESETS

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
DEFAULT_PMNS = PMNSParams()


def pmns_with(theta23_deg=49.2, dm32=2.441e-3, delta_cp_deg=-91.7):
    """Build PMNS parameters from tutorial controls."""
    dm21 = DEFAULT_PMNS.dm21
    return PMNSParams(
        th12=DEFAULT_PMNS.th12,
        th13=DEFAULT_PMNS.th13,
        th23=np.radians(theta23_deg),
        dm21=dm21,
        dm31=dm32 + dm21,
        delta_cp=np.radians(delta_cp_deg),
    )


def oscillation_curves(
    experiment="NOvA",
    theta23_deg=49.2,
    dm32=2.441e-3,
    delta_cp_deg=-91.7,
    n_points=700,
):
    """Compute nu_mu disappearance and nu_e appearance with nuosclab."""
    return compute_curves(
        ExplorerConfig(
            experiment=experiment,
            pmns=pmns_with(theta23_deg, dm32, delta_cp_deg),
            n_points=n_points,
            include_standard=False,
            include_nominal=True,
        )
    )


def beam_shape(energy, peak, width):
    """A smooth toy beam energy spectrum, not an official flux prediction."""
    low_energy_turn_on = 1.0 - np.exp(-((energy / 0.45) ** 2))
    return low_energy_turn_on * np.exp(-0.5 * ((energy - peak) / width) ** 2)


def toy_spectra(
    experiment="NOvA",
    theta23_deg=49.2,
    dm32=2.441e-3,
    delta_cp_deg=-91.7,
    total_near_events=6000,
    n_bins=24,
    seed=7,
    fluctuate=True,
):
    """Return binned near/far toy spectra with optional Poisson fluctuations."""
    preset = PRESETS[experiment]
    edges = np.linspace(*preset.E_range, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    width = 0.65 if experiment == "NOvA" else 0.9
    if experiment == "T2K":
        width = 0.22

    curves = oscillation_curves(
        experiment=experiment,
        theta23_deg=theta23_deg,
        dm32=dm32,
        delta_cp_deg=delta_cp_deg,
        n_points=900,
    )
    survival = np.interp(centers, curves.energy_gev, curves.live[:, 1, 1])
    appearance = np.interp(centers, curves.energy_gev, curves.live[:, 0, 1])

    near_expect = beam_shape(centers, preset.E_peak, width)
    near_expect = total_near_events * near_expect / near_expect.sum()
    far_expect = near_expect * survival

    rng = np.random.default_rng(seed)
    if fluctuate:
        near = rng.poisson(near_expect)
        far = rng.poisson(far_expect)
    else:
        near = near_expect
        far = far_expect

    ratio = np.divide(
        far,
        near,
        out=np.full_like(far, np.nan, dtype=float),
        where=near > 0,
    )
    return {
        "experiment": experiment,
        "edges": edges,
        "centers": centers,
        "near": near,
        "far": far,
        "near_expect": near_expect,
        "far_expect": far_expect,
        "ratio": ratio,
        "survival": survival,
        "appearance": appearance,
        "curves": curves,
    }

In [ ]:
def plot_near_far(result):
    """Make the main tutorial plot: spectra above, far/near ratio below."""
    centers = result["centers"]
    widths = np.diff(result["edges"])
    experiment = result["experiment"]

    fig, (ax_top, ax_ratio) = plt.subplots(
        2,
        1,
        figsize=(9, 7),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1]},
        constrained_layout=True,
    )
    ax_top.bar(
        centers,
        result["near"],
        width=widths,
        alpha=0.45,
        label="Near detector toy data",
        color="#4c78a8",
        edgecolor="black",
        linewidth=0.5,
    )
    ax_top.bar(
        centers,
        result["far"],
        width=widths,
        alpha=0.55,
        label="Far detector toy data",
        color="#f58518",
        edgecolor="black",
        linewidth=0.5,
    )
    ax_top.plot(
        centers,
        result["far_expect"],
        color="#9c3d10",
        linewidth=2,
        label="Expected far spectrum",
    )
    ax_top.set_ylabel("Events per bin")
    ax_top.set_title(f"{experiment} toy near/far spectra")
    ax_top.legend()

    ax_ratio.scatter(centers, result["ratio"], color="#b279a2", zorder=3)
    ax_ratio.plot(
        centers,
        result["survival"],
        color="black",
        linewidth=2,
        label=r"$P(\nu_\mu \to \nu_\mu)$",
    )
    ax_ratio.set_xlabel("Reconstructed neutrino energy (GeV)")
    ax_ratio.set_ylabel("Far / near")
    ax_ratio.set_ylim(0, 1.15)
    ax_ratio.legend(loc="lower right")
    return fig


def show_spectra(
    experiment="NOvA",
    theta23_deg=49.2,
    dm32_x1000=2.441,
    total_near_events=6000,
    n_bins=24,
    seed=7,
    fluctuate=True,
):
    result = toy_spectra(
        experiment=experiment,
        theta23_deg=theta23_deg,
        dm32=dm32_x1000 * 1e-3,
        total_near_events=total_near_events,
        n_bins=n_bins,
        seed=seed,
        fluctuate=fluctuate,
    )
    plot_near_far(result)
    plt.show()

## 1. Compare smooth probability curves

First look at the probability itself, without detector statistics. This is the cleanest way to see how the parameters change the oscillation pattern.


In [ ]:
def plot_probability_controls(
    experiment="NOvA",
    theta23_deg=49.2,
    dm32_x1000=2.441,
    delta_cp_deg=-91.7,
):
    curves = oscillation_curves(
        experiment=experiment,
        theta23_deg=theta23_deg,
        dm32=dm32_x1000 * 1e-3,
        delta_cp_deg=delta_cp_deg,
    )
    energy = curves.energy_gev
    survival = curves.live[:, 1, 1]
    appearance = curves.live[:, 0, 1]

    fig, ax = plt.subplots(figsize=(9, 4.8), constrained_layout=True)
    ax.plot(energy, survival, label=r"survival $P(\nu_\mu \to \nu_\mu)$")
    ax.plot(energy, appearance, label=r"appearance $P(\nu_\mu \to \nu_e)$")
    ax.axvline(curves.preset.E_peak, color="0.4", linestyle=":", label="beam peak")
    ax.set_title(f"{experiment} oscillation probabilities")
    ax.set_xlabel("Neutrino energy (GeV)")
    ax.set_ylabel("Probability")
    ax.set_ylim(-0.02, 1.05)
    ax.legend()
    plt.show()


widgets.interact(
    plot_probability_controls,
    experiment=widgets.ToggleButtons(options=["NOvA", "DUNE", "T2K"], value="NOvA"),
    theta23_deg=widgets.FloatSlider(value=49.2, min=40.0, max=55.0, step=0.2),
    dm32_x1000=widgets.FloatSlider(value=2.441, min=2.1, max=2.8, step=0.01),
    delta_cp_deg=widgets.FloatSlider(value=-91.7, min=-180.0, max=180.0, step=5.0),
)

## 2. Add the toy detector view

A measurement sees binned event counts. Use this control to compare the clean probability curve with one possible fluctuated dataset.


In [ ]:
widgets.interact(
    show_spectra,
    experiment=widgets.ToggleButtons(options=["NOvA", "DUNE"], value="NOvA"),
    theta23_deg=widgets.FloatSlider(value=49.2, min=40.0, max=55.0, step=0.2),
    dm32_x1000=widgets.FloatSlider(value=2.441, min=2.1, max=2.8, step=0.01),
    total_near_events=widgets.IntSlider(value=6000, min=500, max=20000, step=500),
    n_bins=widgets.IntSlider(value=24, min=10, max=50, step=2),
    seed=widgets.IntSlider(value=7, min=1, max=999, step=1),
    fluctuate=widgets.Checkbox(value=True),
)

## 3. A simple parameter scan

Now make one toy dataset and scan possible Δm²₃₂ values. This is not a full experimental fit, but it shows the core idea: predictions that put the dip in the wrong place get a worse score.


In [ ]:
observed = toy_spectra(
    experiment="NOvA",
    theta23_deg=49.2,
    dm32=2.441e-3,
    total_near_events=8000,
    n_bins=24,
    seed=14,
    fluctuate=True,
)


def chi2_for_dm32(dm32):
    prediction = toy_spectra(
        experiment="NOvA",
        theta23_deg=49.2,
        dm32=dm32,
        total_near_events=8000,
        n_bins=24,
        seed=14,
        fluctuate=False,
    )
    expected = np.clip(prediction["far_expect"], 1.0, None)
    return np.sum((observed["far"] - expected) ** 2 / expected)


dm32_grid = np.linspace(2.1e-3, 2.8e-3, 81)
chi2_values = np.array([chi2_for_dm32(value) for value in dm32_grid])
best_index = np.argmin(chi2_values)
best_dm32 = dm32_grid[best_index]

fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
ax.plot(dm32_grid * 1e3, chi2_values, marker="o", markersize=3)
ax.axvline(best_dm32 * 1e3, color="black", linestyle="--", label="best scan point")
ax.set_xlabel(r"$\Delta m^2_{32}$ ($10^{-3}$ eV$^2$)")
ax.set_ylabel("Toy chi-square")
ax.set_title("Simple scan over the oscillation frequency")
ax.legend()
plt.show()

print(f"Best scan point: Delta m^2_32 = {best_dm32:.4e} eV^2")

## 4. Exercises

1. Change the random seed in the toy dataset. Does the best-fit point move? Why?
2. Increase `total_near_events` from 8,000 to 20,000 in both places. What happens to the scan curve?
3. Change the true value used to make `observed`. Does the scan recover the new value?
4. Extension: make a second scan over θ₂₃ and compare what θ₂₃ changes versus what Δm²₃₂ changes.
5. Finished early? Try the bonus notebook `bonus_nsi_explorer.ipynb`. It asks: what if there's a hidden new-physics effect the standard three-flavor picture doesn't capture?


In [ ]:
# Exercise workspace: start by changing these values and rerunning the scan above.
true_dm32 = 2.441e-3
true_theta23 = 49.2
sample_size = 8000
random_seed = 14

print("Try editing:")
print("true_dm32 =", true_dm32)
print("true_theta23 =", true_theta23)
print("sample_size =", sample_size)
print("random_seed =", random_seed)

## 5. Poster connection

For a poster, do not show every slider. Choose one plot that supports one sentence.

A strong sentence has this shape: **what you compared + what you saw + what it means**.

Here are a few starting points — pick whichever matches *your* plot, or write your own in the same shape:

> We compared the expected near-detector spectrum with the far-detector spectrum and saw a dip, which is evidence that muon neutrinos changed flavor while traveling.

> By scanning Δm²₃₂ against our toy dataset, the best-fit value landed close to the input we used to generate the data, showing that the dip's position encodes the neutrino mass-squared splitting.

> When we compared how the dip moved as we changed θ₂₃ versus Δm²₃₂, we saw that one parameter changes the depth of the dip and the other changes where it sits — they answer different physics questions.

Don't just copy one of these. Look at the plot you're actually proudest of, and write the sentence that plot earns.
